# Exercise 3: Load Balancing Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

## 1. Workload Distribution

In [ ]:
# Workload ratios
naive_workload = {
    'Section 1 (Light)': 1,
    'Section 2 (Moderate)': 5,
    'Section 3 (Heavy)': 20
}

optimized_workload = {
    'Section 1 (Light+Mod)': 6,
    'Section 2 (Heavy 1/3)': 6.67,
    'Section 3 (Heavy 2/3)': 6.67,
    'Section 4 (Heavy 3/3)': 6.67
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Naive
colors1 = ['#3498db', '#f39c12', '#e74c3c']
ax1.barh(list(naive_workload.keys()), list(naive_workload.values()), 
         color=colors1, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Relative Work Units', fontsize=12)
ax1.set_title('Naive: Imbalanced (1:5:20)', fontsize=14, weight='bold')
ax1.grid(axis='x', alpha=0.3)

# Add load balance score
avg_naive = np.mean(list(naive_workload.values()))
max_naive = max(naive_workload.values())
efficiency_naive = avg_naive / max_naive
ax1.text(0.98, 0.02, f'Efficiency: {efficiency_naive:.1%}', 
         transform=ax1.transAxes, ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
         fontsize=11, weight='bold')

# Optimized
colors2 = ['#3498db', '#2ecc71', '#2ecc71', '#2ecc71']
ax2.barh(list(optimized_workload.keys()), list(optimized_workload.values()),
         color=colors2, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Relative Work Units', fontsize=12)
ax2.set_title('Optimized: Balanced (6:7:7:7)', fontsize=14, weight='bold')
ax2.grid(axis='x', alpha=0.3)

# Add load balance score
avg_opt = np.mean(list(optimized_workload.values()))
max_opt = max(optimized_workload.values())
efficiency_opt = avg_opt / max_opt
ax2.text(0.98, 0.02, f'Efficiency: {efficiency_opt:.1%}',
         transform=ax2.transAxes, ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
         fontsize=11, weight='bold')

plt.tight_layout()
plt.savefig('ex3_workload.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Naive efficiency: {efficiency_naive:.1%}")
print(f"Optimized efficiency: {efficiency_opt:.1%}")
print(f"Improvement: {(efficiency_opt/efficiency_naive - 1)*100:.1f}%")

## 2. Timeline Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Naive timeline
tasks_naive = [
    {'thread': 0, 'start': 0, 'end': 1, 'label': 'Light (1x)', 'color': '#3498db'},
    {'thread': 1, 'start': 0, 'end': 5, 'label': 'Moderate (5x)', 'color': '#f39c12'},
    {'thread': 2, 'start': 0, 'end': 20, 'label': 'Heavy (20x)', 'color': '#e74c3c'},
]

for task in tasks_naive:
    ax1.barh(task['thread'], task['end'] - task['start'], 
             left=task['start'], height=0.8,
             color=task['color'], edgecolor='black', linewidth=1.5)
    ax1.text(task['start'] + (task['end'] - task['start'])/2, task['thread'],
             task['label'], ha='center', va='center', fontsize=10, weight='bold')

# Mark idle time
ax1.barh(0, 19, left=1, height=0.8, color='lightgray', alpha=0.5, hatch='///')
ax1.barh(1, 15, left=5, height=0.8, color='lightgray', alpha=0.5, hatch='///')

ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['Thread 0', 'Thread 1', 'Thread 2'])
ax1.set_xlabel('Time Units', fontsize=12)
ax1.set_title('Naive: Poor Load Balance (threads idle)', fontsize=14, weight='bold')
ax1.set_xlim(0, 21)
ax1.grid(axis='x', alpha=0.3)

# Optimized timeline
tasks_opt = [
    {'thread': 0, 'start': 0, 'end': 6, 'label': 'Light+Mod (6x)', 'color': '#3498db'},
    {'thread': 1, 'start': 0, 'end': 6.67, 'label': 'Heavy 1/3', 'color': '#2ecc71'},
    {'thread': 2, 'start': 0, 'end': 6.67, 'label': 'Heavy 2/3', 'color': '#2ecc71'},
    {'thread': 3, 'start': 0, 'end': 6.67, 'label': 'Heavy 3/3', 'color': '#2ecc71'},
]

for task in tasks_opt:
    ax2.barh(task['thread'], task['end'] - task['start'],
             left=task['start'], height=0.8,
             color=task['color'], edgecolor='black', linewidth=1.5)
    ax2.text(task['start'] + (task['end'] - task['start'])/2, task['thread'],
             task['label'], ha='center', va='center', fontsize=10, weight='bold')

ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['Thread 0', 'Thread 1', 'Thread 2', 'Thread 3'])
ax2.set_xlabel('Time Units', fontsize=12)
ax2.set_title('Optimized: Good Load Balance (minimal idle)', fontsize=14, weight='bold')
ax2.set_xlim(0, 21)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('ex3_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Performance Measurements

In [ ]:
# Example data - replace with your measurements
data = {
    'version': ['Serial', 'Naive (1T)', 'Naive (3T)', 'Naive (4T)', 
                'Optimized (1T)', 'Optimized (4T)', 'Optimized (8T)'],
    'time': [1.50, 1.52, 0.77, 0.77, 1.52, 0.40, 0.40],  # Replace with actual
    'threads': [1, 1, 3, 4, 1, 4, 8]
}

df = pd.DataFrame(data)
baseline = df[df['version'] == 'Serial']['time'].values[0]
df['speedup'] = baseline / df['time']
df['efficiency'] = (df['speedup'] / df['threads']) * 100

print(df)
print(f"\nBest speedup: {df['speedup'].max():.2f}x with {df.loc[df['speedup'].idxmax(), 'version']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Execution time comparison
x = np.arange(len(df))
colors = ['gray', 'orange', 'orange', 'orange', 'lightblue', 'green', 'green']
bars = axes[0].bar(x, df['time'], color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df['version'], rotation=45, ha='right')
axes[0].set_ylabel('Time (s)', fontsize=12)
axes[0].set_title('Execution Time Comparison', fontsize=14, weight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add time labels
for i, (bar, time) in enumerate(zip(bars, df['time'])):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{time:.2f}s', ha='center', va='bottom', fontsize=9)

# Speedup comparison
bars2 = axes[1].bar(x, df['speedup'], color=colors, edgecolor='black', linewidth=1.5)
axes[1].axhline(y=1, linestyle='--', color='gray', label='Serial baseline')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df['version'], rotation=45, ha='right')
axes[1].set_ylabel('Speedup', fontsize=12)
axes[1].set_title('Speedup vs Serial', fontsize=14, weight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend()

# Add speedup labels
for i, (bar, speedup) in enumerate(zip(bars2, df['speedup'])):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{speedup:.2f}x', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('ex3_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Key Takeaways

### Load Imbalance Impact:
- **Naive**: 1:5:20 ratio → only 41% efficiency
- **Optimized**: 6:7:7:7 ratio → 95% efficiency
- **Improvement**: ~2.3x better load balance

### Optimization Strategies:
1. **Merge light tasks** to avoid thread underutilization
2. **Split heavy tasks** to distribute work evenly
3. **Aim for equal workload** across all sections

### When Sections Scale Poorly:
- Fixed number of sections limits parallelism
- No dynamic load balancing
- Consider using `tasks` or `parallel for` instead